In [ ]:
"""Computes monthly baseflow using seven hydrograph-separation methods and derives
BFI, DSF, additive BBI (DSF - BFI), threshold sensitivity, severity classes,
baseflow retention, and normalized-Q20 deficit diagnostics. Results are written
to a single NetCDF file (This codd is refined with .
"""
from __future__ import annotations

import gc
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from netCDF4 import Dataset
from numba import njit

# Set input and output locations here.
IN_NC = Path("data/flow.nc")
OUT_DIR = Path("outputs/bbi_reanalysis")
OUT_NC = OUT_DIR / "bbi_threshold_severity_analysis.nc"

FLOW_VAR = "flow_mean"
TIME_NAME, LAT_NAME, LON_NAME = "time", "lat", "lon"

# Threshold climatology and BFI baseline.
BASELINE_START = "1981-01-01"
BASELINE_END = "2022-12-31"

ANALYSIS_START = "1981-01-01"
ANALYSIS_END = "2022-12-31"

THRESHOLD_PCTS = np.array([5, 10, 20, 30], dtype=np.int16)
REFERENCE_THRESHOLD = 20

# S20 = (Q20-Q)/Q20 is bounded by [0,1] for nonnegative discharge.
DEFICIT_BIN_EDGES = np.array([0.0, 0.10, 0.25, 0.50, 1.000001], dtype=np.float64)
DEFICIT_BIN_LABELS = (
    "[0,0.1)",
    "[0.1,0.25)",
    "[0.25,0.5)",
    "[0.5,1]",
)
SEVERITY_CLASS_NAMES = (
    "moderate_Q10_Q20",
    "severe_Q5_Q10",
    "extreme_below_Q5",
)

# Data sufficiency rules.
MIN_DAILY_COVERAGE = 0.90
MIN_BASELINE_SAMPLES_PER_CAL_MONTH = 20
MIN_BASELINE_MONTHS_FOR_BFI = 240
MIN_DROUGHT_MONTHS = 10
MIN_DEFICIT_BIN_MONTHS = 5
MIN_VALID_DEFICIT_BINS_FOR_TREND = 3
MIN_PRIMARY_METHODS = 5       # at least 5 of 7 unique methods
MIN_LEGACY_METHODS = 5        # at least 5 of 9 legacy members

# Numerical settings.
EPS = 1.0e-12
ECK_ALPHA = 0.98
ECK_BFI_MAX = 0.80
LH_ALPHA = 0.925
LH_REFLECT_DAYS = 30
CHAP_ALPHA = 0.925
HYSEP_N_DAYS = 5

# Block size for processing. Reduce if RAM is limited.
LAT_BLOCK = 20
LON_BLOCK = 30
COMPLEVEL = 4


# METHOD SETS

PRIMARY_METHOD_NAMES = (
    "eckhardt",
    "lyne_hollick_1pass",
    "lyne_hollick_3pass",
    "chapman",
    "hysep_fixed_interval",
    "hysep_sliding_interval",
    "hysep_local_minimum",
)

# Retained only to quantify sensitivity.
LEGACY_METHOD_NAMES = (
    "eckhardt",
    "lyne_hollick_1pass",
    "lyne_hollick_3pass",
    "chapman",
    "fixed_frac_0p5",
    "fixed_frac_0p7",
    "local_minimum_window",
    "hysep_fixed_interval",
    "hysep_sliding_duplicate",
)

# =============================================================================
def ensure_output_directory() -> None:
    """Create the output directory."""
    OUT_DIR.mkdir(parents=True, exist_ok=True)


def decode_time(ds: xr.Dataset) -> pd.DatetimeIndex:
    """Decode a CF-compliant or simple numeric time coordinate."""
    try:
        decoded = xr.decode_cf(ds)
        values = decoded[TIME_NAME].values
        return pd.DatetimeIndex(values)
    except Exception:
        units = ds[TIME_NAME].attrs.get(
            "units", "days since 1975-01-01 00:00:00"
        )
        if "since" not in units:
            raise ValueError(f"Cannot decode time units: {units!r}")
        base_text = units.split("since", 1)[1].strip()
        base = pd.Timestamp(base_text)
        values = np.asarray(ds[TIME_NAME].values, dtype=np.float64)
        unit = "D"
        lower = units.lower()
        if lower.startswith("hours"):
            unit = "h"
        elif lower.startswith("minutes"):
            unit = "m"
        elif lower.startswith("seconds"):
            unit = "s"
        return pd.DatetimeIndex(base + pd.to_timedelta(values, unit=unit))


def validate_time(time: pd.DatetimeIndex) -> None:
    if time.hasnans:
        raise ValueError("Time coordinate contains NaT values.")
    if time.has_duplicates:
        raise ValueError("Time coordinate contains duplicate timestamps.")
    if not time.is_monotonic_increasing:
        raise ValueError("Time coordinate must be monotonically increasing.")


def safe_ratio(numerator: np.ndarray, denominator: np.ndarray) -> np.ndarray:
    shape = np.broadcast_shapes(numerator.shape, denominator.shape)
    out = np.full(shape, np.nan, dtype=np.float32)
    valid = (
        np.isfinite(numerator)
        & np.isfinite(denominator)
        & (np.abs(denominator) > EPS)
    )
    np.divide(numerator, denominator, out=out, where=valid)
    return out


def period_mask(months: pd.PeriodIndex, start: str, end: str) -> np.ndarray:
    """Inclusive month mask, robust to day-level start/end strings."""
    start_period = pd.Period(pd.Timestamp(start), freq="M")
    end_period = pd.Period(pd.Timestamp(end), freq="M")
    return np.asarray((months >= start_period) & (months <= end_period), dtype=bool)


# BASEFLOW SEPARATION
# BASEFLOW SEPARATION
@njit(cache=True)
def _clip_baseflow(qb: float, q: float) -> float:
    if qb < 0.0:
        qb = 0.0
    if qb > q:
        qb = q
    return qb


@njit(cache=True)
def baseflow_eckhardt(Q: np.ndarray, alpha: float, bfi_max: float) -> np.ndarray:
    T, Y, X = Q.shape
    Qb = np.full_like(Q, np.nan)
    c = (1.0 - bfi_max) * alpha
    denominator = 1.0 - alpha * bfi_max

    for j in range(Y):
        for i in range(X):
            for t in range(T):
                q = Q[t, j, i]
                if np.isnan(q):
                    continue
                if t == 0 or np.isnan(Qb[t - 1, j, i]):
                    Qb[t, j, i] = bfi_max * q
                    continue
                qb = (
                    c * Qb[t - 1, j, i]
                    + (1.0 - alpha) * bfi_max * q
                ) / denominator
                Qb[t, j, i] = _clip_baseflow(qb, q)
    return Qb


@njit(cache=True)
def _lh_first_pass_1d(q: np.ndarray, alpha: float) -> np.ndarray:
    """First Lyne-Hollick pass following Ladson et al. (2013)."""
    n = q.size
    qb = np.empty(n, dtype=np.float32)
    qf_prev = q[0]
    qb[0] = q[0] - qf_prev if qf_prev > 0.0 else q[0]
    for t in range(1, n):
        qf = alpha * qf_prev + 0.5 * (1.0 + alpha) * (q[t] - q[t - 1])
        qb[t] = q[t] - qf if qf > 0.0 else q[t]
        qf_prev = qf
    return qb


@njit(cache=True)
def _lh_backward_pass_1d(qb_in: np.ndarray, alpha: float) -> np.ndarray:
    """Backward Lyne-Hollick pass applied to baseflow from the preceding pass."""
    n = qb_in.size
    qb_out = np.empty(n, dtype=np.float32)
    qf_next = qb_in[n - 1]
    qb_out[n - 1] = qb_in[n - 1] - qf_next if qf_next > 0.0 else qb_in[n - 1]
    for t in range(n - 2, -1, -1):
        qf = alpha * qf_next + 0.5 * (1.0 + alpha) * (
            qb_in[t] - qb_in[t + 1]
        )
        qb_out[t] = qb_in[t] - qf if qf > 0.0 else qb_in[t]
        qf_next = qf
    return qb_out


@njit(cache=True)
def _lh_forward_pass_1d(qb_in: np.ndarray, alpha: float) -> np.ndarray:
    """Forward Lyne-Hollick pass applied to baseflow from the preceding pass."""
    n = qb_in.size
    qb_out = np.empty(n, dtype=np.float32)
    qf_prev = qb_in[0]
    qb_out[0] = qb_in[0] - qf_prev if qf_prev > 0.0 else qb_in[0]
    for t in range(1, n):
        qf = alpha * qf_prev + 0.5 * (1.0 + alpha) * (
            qb_in[t] - qb_in[t - 1]
        )
        qb_out[t] = qb_in[t] - qf if qf > 0.0 else qb_in[t]
        qf_prev = qf
    return qb_out


@njit(cache=True)
def _lh_segment(
    q: np.ndarray,
    alpha: float,
    passes: int,
    reflect_days: int,
) -> np.ndarray:
    """Filter one finite segment using reflected endpoints."""
    n = q.size
    nref = reflect_days
    if nref > n - 1:
        nref = n - 1

    q_ref = np.empty(n + 2 * nref, dtype=np.float32)
    # Reflect values adjacent to, but not including, the endpoints.
    for r in range(nref):
        q_ref[r] = q[nref - r]
    for t in range(n):
        q_ref[nref + t] = q[t]
    for r in range(nref):
        q_ref[nref + n + r] = q[n - 2 - r]

    qb = _lh_first_pass_1d(q_ref, alpha)
    if passes >= 3:
        n_pairs = (passes - 1) // 2
        for _ in range(n_pairs):
            qb = _lh_backward_pass_1d(qb, alpha)
            qb = _lh_forward_pass_1d(qb, alpha)

    out = np.empty(n, dtype=np.float32)
    for t in range(n):
        value = qb[nref + t]
        if value < 0.0:
            value = 0.0
        if value > q[t]:
            value = q[t]
        out[t] = value
    return out


@njit(cache=True)
def baseflow_lyne_hollick(
    Q: np.ndarray,
    alpha: float,
    passes: int,
    reflect_days: int,
) -> np.ndarray:
    """Lyne-Hollick filtering of each contiguous finite discharge segment."""
    T, Y, X = Q.shape
    Qb = np.full_like(Q, np.nan)
    for j in range(Y):
        for i in range(X):
            start = 0
            while start < T:
                while start < T and np.isnan(Q[start, j, i]):
                    start += 1
                if start >= T:
                    break
                end = start
                while end < T and not np.isnan(Q[end, j, i]):
                    end += 1
                length = end - start
                # Ladson et al. require a segment longer than the reflection length.
                if length > reflect_days:
                    segment = np.empty(length, dtype=np.float32)
                    for t in range(length):
                        segment[t] = Q[start + t, j, i]
                    filtered = _lh_segment(segment, alpha, passes, reflect_days)
                    for t in range(length):
                        Qb[start + t, j, i] = filtered[t]
                start = end + 1
    return Qb


@njit(cache=True)
def baseflow_lyne_hollick_onepass(Q: np.ndarray, alpha: float) -> np.ndarray:
    return baseflow_lyne_hollick(Q, alpha, 1, LH_REFLECT_DAYS)


@njit(cache=True)
def baseflow_lyne_hollick_3pass(Q: np.ndarray, alpha: float) -> np.ndarray:
    return baseflow_lyne_hollick(Q, alpha, 3, LH_REFLECT_DAYS)


@njit(cache=True)
def baseflow_chapman(Q: np.ndarray, alpha: float) -> np.ndarray:
    T, Y, X = Q.shape
    quick = np.full_like(Q, np.nan)
    a = (3.0 * alpha - 1.0) / (3.0 - alpha)
    b = (1.0 - alpha) / (3.0 - alpha)

    for j in range(Y):
        for i in range(X):
            for t in range(T):
                q = Q[t, j, i]
                if np.isnan(q):
                    continue
                if (
                    t == 0
                    or np.isnan(Q[t - 1, j, i])
                    or np.isnan(quick[t - 1, j, i])
                ):
                    quick[t, j, i] = 0.0
                    continue
                q_prev = Q[t - 1, j, i]
                qf = a * quick[t - 1, j, i] + b * (q - q_prev)
                if qf < 0.0:
                    qf = 0.0
                if qf > q:
                    qf = q
                quick[t, j, i] = qf

    Qb = Q - quick
    for t in range(T):
        for j in range(Y):
            for i in range(X):
                q = Q[t, j, i]
                qb = Qb[t, j, i]
                if not np.isnan(q) and not np.isnan(qb):
                    Qb[t, j, i] = _clip_baseflow(qb, q)
    return Qb


@njit(cache=True)
def baseflow_hysep_sliding_interval(Q: np.ndarray, n_days: int) -> np.ndarray:
    """USGS HYSEP sliding-interval method."""
    T, Y, X = Q.shape
    Qb = np.full_like(Q, np.nan)
    half = (n_days - 1) // 2
    for t in range(T):
        t0 = max(0, t - half)
        t1 = min(T, t + half + 1)
        for j in range(Y):
            for i in range(X):
                q_now = Q[t, j, i]
                if np.isnan(q_now):
                    continue
                minimum = np.inf
                for k in range(t0, t1):
                    q = Q[k, j, i]
                    if not np.isnan(q) and q < minimum:
                        minimum = q
                if minimum != np.inf:
                    Qb[t, j, i] = _clip_baseflow(minimum, q_now)
    return Qb


@njit(cache=True)
def baseflow_hysep_fixed_interval(Q: np.ndarray, n_days: int) -> np.ndarray:
    """USGS HYSEP fixed-interval method."""
    T, Y, X = Q.shape
    Qb = np.full_like(Q, np.nan)
    for start in range(0, T, n_days):
        end = min(T, start + n_days)
        for j in range(Y):
            for i in range(X):
                minimum = np.inf
                for t in range(start, end):
                    q = Q[t, j, i]
                    if not np.isnan(q) and q < minimum:
                        minimum = q
                if minimum == np.inf:
                    continue
                for t in range(start, end):
                    q = Q[t, j, i]
                    if not np.isnan(q):
                        Qb[t, j, i] = _clip_baseflow(minimum, q)
    return Qb


@njit(cache=True)
def baseflow_hysep_local_minimum(Q: np.ndarray, n_days: int) -> np.ndarray:
    """USGS HYSEP local-minimum method with linear interpolation."""
    T, Y, X = Q.shape
    Qb = np.full_like(Q, np.nan)
    half = (n_days - 1) // 2

    for j in range(Y):
        for i in range(X):
            start = 0
            while start < T:
                while start < T and np.isnan(Q[start, j, i]):
                    start += 1
                if start >= T:
                    break
                end = start
                while end < T and not np.isnan(Q[end, j, i]):
                    end += 1

                length = end - start
                anchors = np.empty(length, dtype=np.int64)
                n_anchor = 0

                for t in range(start, end):
                    t0 = max(start, t - half)
                    t1 = min(end, t + half + 1)
                    minimum = np.inf
                    for k in range(t0, t1):
                        q = Q[k, j, i]
                        if q < minimum:
                            minimum = q
                    if Q[t, j, i] <= minimum:
                        # Collapse flat consecutive minima into one central anchor.
                        if (
                            n_anchor > 0
                            and t == anchors[n_anchor - 1] + 1
                            and Q[t, j, i] == Q[anchors[n_anchor - 1], j, i]
                        ):
                            anchors[n_anchor - 1] = t
                        else:
                            anchors[n_anchor] = t
                            n_anchor += 1

                if n_anchor == 0:
                    minimum = np.inf
                    for t in range(start, end):
                        q = Q[t, j, i]
                        if q < minimum:
                            minimum = q
                    for t in range(start, end):
                        Qb[t, j, i] = _clip_baseflow(minimum, Q[t, j, i])
                else:
                    first = anchors[0]
                    first_value = Q[first, j, i]
                    for t in range(start, first + 1):
                        Qb[t, j, i] = _clip_baseflow(first_value, Q[t, j, i])

                    for a in range(n_anchor - 1):
                        left = anchors[a]
                        right = anchors[a + 1]
                        q_left = Q[left, j, i]
                        q_right = Q[right, j, i]
                        span = right - left
                        for t in range(left, right + 1):
                            fraction = (t - left) / span
                            value = q_left + fraction * (q_right - q_left)
                            Qb[t, j, i] = _clip_baseflow(value, Q[t, j, i])

                    last = anchors[n_anchor - 1]
                    last_value = Q[last, j, i]
                    for t in range(last, end):
                        Qb[t, j, i] = _clip_baseflow(last_value, Q[t, j, i])

                start = end + 1

    return Qb


def compute_primary_method(Q: np.ndarray, method: str) -> np.ndarray:
    if method == "eckhardt":
        return baseflow_eckhardt(Q, ECK_ALPHA, ECK_BFI_MAX)
    if method == "lyne_hollick_1pass":
        return baseflow_lyne_hollick_onepass(Q, LH_ALPHA)
    if method == "lyne_hollick_3pass":
        return baseflow_lyne_hollick_3pass(Q, LH_ALPHA)
    if method == "chapman":
        return baseflow_chapman(Q, CHAP_ALPHA)
    if method == "hysep_fixed_interval":
        return baseflow_hysep_fixed_interval(Q, HYSEP_N_DAYS)
    if method == "hysep_sliding_interval":
        return baseflow_hysep_sliding_interval(Q, HYSEP_N_DAYS)
    if method == "hysep_local_minimum":
        return baseflow_hysep_local_minimum(Q, HYSEP_N_DAYS)
    raise KeyError(f"Unknown method: {method}")

# MONTHLY PROCESSING

def daily_to_monthly_sums(
    values: np.ndarray,
    months_daily: pd.PeriodIndex,
) -> np.ndarray:
    unique_months = months_daily.unique()
    _, Y, X = values.shape
    out = np.full((len(unique_months), Y, X), np.nan, dtype=np.float32)

    for m, period in enumerate(unique_months):
        indices = np.flatnonzero(months_daily == period)
        block = values[indices, :, :]
        valid_count = np.sum(np.isfinite(block), axis=0)
        required = max(1, int(math.ceil(MIN_DAILY_COVERAGE * indices.size)))
        total = np.nansum(block, axis=0, dtype=np.float64)
        total[valid_count < required] = np.nan
        out[m] = total.astype(np.float32)
    return out


def calendar_month_percentiles(
    Qm: np.ndarray,
    months: pd.PeriodIndex,
    baseline_mask: np.ndarray,
) -> np.ndarray:
    _, Y, X = Qm.shape
    out = np.full((len(THRESHOLD_PCTS), 12, Y, X), np.nan, dtype=np.float32)

    for calendar_month in range(1, 13):
        selected = baseline_mask & (months.month == calendar_month)
        data = Qm[selected]
        if data.shape[0] == 0:
            continue
        counts = np.sum(np.isfinite(data), axis=0)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            try:
                values = np.nanpercentile(
                    data, THRESHOLD_PCTS, axis=0, method="linear"
                )
            except TypeError:  # compatibility with older NumPy
                values = np.nanpercentile(
                    data, THRESHOLD_PCTS, axis=0, interpolation="linear"
                )
        values = values.astype(np.float32)
        values[:, counts < MIN_BASELINE_SAMPLES_PER_CAL_MONTH] = np.nan
        out[:, calendar_month - 1] = values
    return out


def calendar_month_mean(
    values: np.ndarray,
    months: pd.PeriodIndex,
    baseline_mask: np.ndarray,
) -> np.ndarray:
    _, Y, X = values.shape
    out = np.full((12, Y, X), np.nan, dtype=np.float32)
    for calendar_month in range(1, 13):
        selected = baseline_mask & (months.month == calendar_month)
        data = values[selected]
        count = np.sum(np.isfinite(data), axis=0)
        total = np.nansum(data, axis=0, dtype=np.float64)
        valid = count >= MIN_BASELINE_SAMPLES_PER_CAL_MONTH
        mean = np.full((Y, X), np.nan, dtype=np.float64)
        mean[valid] = total[valid] / count[valid]
        out[calendar_month - 1] = mean.astype(np.float32)
    return out


def expand_calendar_month_field(
    field12: np.ndarray, months: pd.PeriodIndex
) -> np.ndarray:
    return np.stack([field12[m - 1] for m in months.month], axis=0).astype(np.float32)


def ensemble_median(
    stack: np.ndarray, minimum_methods: int
) -> tuple[np.ndarray, np.ndarray]:
    count = np.sum(np.isfinite(stack), axis=0).astype(np.int8)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        median = np.nanmedian(stack, axis=0).astype(np.float32)
    median[count < minimum_methods] = np.nan
    return median, count


def calculate_bfi(
    Qbm: np.ndarray,
    Qm: np.ndarray,
    baseline_mask: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    valid = (
        baseline_mask[:, None, None]
        & np.isfinite(Qbm)
        & np.isfinite(Qm)
        & (Qm > EPS)
    )
    count = np.sum(valid, axis=0).astype(np.int16)
    sum_q = np.sum(np.where(valid, Qm, 0.0), axis=0, dtype=np.float64)
    sum_qb = np.sum(np.where(valid, Qbm, 0.0), axis=0, dtype=np.float64)
    bfi = safe_ratio(sum_qb, sum_q)
    bfi[count < MIN_BASELINE_MONTHS_FOR_BFI] = np.nan
    return bfi, count
# DROUGHT METRICS

def weighted_dsf(
    Qbm: np.ndarray,
    Qm: np.ndarray,
    mask: np.ndarray,
    minimum_months: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Zero-flow months are drought months but Qb/Q is undefined at Q=0.
    used = mask & np.isfinite(Qbm) & np.isfinite(Qm) & (Qm > EPS)
    total_drought = mask & np.isfinite(Qm) & (Qm >= 0.0)
    count_used = np.sum(used, axis=0).astype(np.int16)
    count_total = np.sum(total_drought, axis=0).astype(np.int16)
    sum_q = np.sum(np.where(used, Qm, 0.0), axis=0, dtype=np.float64)
    sum_qb = np.sum(np.where(used, Qbm, 0.0), axis=0, dtype=np.float64)
    dsf = safe_ratio(sum_qb, sum_q)
    dsf[count_used < minimum_months] = np.nan
    return dsf, count_used, count_total


def baseflow_retention_ratio(
    Qbm: np.ndarray,
    expected_qb: np.ndarray,
    mask: np.ndarray,
    minimum_months: int,
) -> np.ndarray:
    valid = (
        mask
        & np.isfinite(Qbm)
        & np.isfinite(expected_qb)
        & (expected_qb > EPS)
    )
    count = np.sum(valid, axis=0)
    actual = np.sum(np.where(valid, Qbm, 0.0), axis=0, dtype=np.float64)
    expected = np.sum(np.where(valid, expected_qb, 0.0), axis=0, dtype=np.float64)
    brr = safe_ratio(actual, expected)
    brr[count < minimum_months] = np.nan
    return brr


def drought_frequency(mask: np.ndarray, eligible: np.ndarray) -> np.ndarray:
    denominator = np.sum(eligible, axis=0)
    numerator = np.sum(mask & eligible, axis=0)
    out = safe_ratio(numerator.astype(np.float64), denominator.astype(np.float64))
    out[denominator == 0] = np.nan
    return out


def make_severity_masks(
    Qm: np.ndarray,
    months: pd.PeriodIndex,
    threshold_map: dict[int, np.ndarray],
    analysis_mask: np.ndarray,
) -> dict[str, np.ndarray]:
    q5 = expand_calendar_month_field(threshold_map[5], months)
    q10 = expand_calendar_month_field(threshold_map[10], months)
    q20 = expand_calendar_month_field(threshold_map[20], months)
    valid = (
        analysis_mask[:, None, None]
        & np.isfinite(Qm)
        & (Qm >= 0.0)
        & np.isfinite(q5)
        & np.isfinite(q10)
        & np.isfinite(q20)
    )
    return {
        "moderate_Q10_Q20": valid & (Qm >= q10) & (Qm < q20),
        "severe_Q5_Q10": valid & (Qm >= q5) & (Qm < q10),
        "extreme_below_Q5": valid & (Qm < q5),
    }


def deficit_bin_metrics(
    Qm: np.ndarray,
    Qbm: np.ndarray,
    BFI: np.ndarray,
    expected_qb: np.ndarray,
    months: pd.PeriodIndex,
    q20_12: np.ndarray,
    analysis_mask: np.ndarray,
) -> dict[str, np.ndarray]:
    q20 = expand_calendar_month_field(q20_12, months)
    valid_q20 = (
        analysis_mask[:, None, None]
        & np.isfinite(Qm)
        & np.isfinite(Qbm)
        & np.isfinite(q20)
        & (q20 > EPS)
        & (Qm >= 0.0)
        & (Qm < q20)
    )

    S20 = np.full(Qm.shape, np.nan, dtype=np.float32)
    S20[valid_q20] = ((q20[valid_q20] - Qm[valid_q20]) / q20[valid_q20]).astype(
        np.float32
    )
    # Protect against very small floating-point excursions.
    S20[(S20 < 0.0) | (S20 > 1.000001)] = np.nan

    B = len(DEFICIT_BIN_LABELS)
    _, Y, X = Qm.shape
    dsf = np.full((B, Y, X), np.nan, dtype=np.float32)
    bbi = np.full_like(dsf, np.nan)
    brr = np.full_like(dsf, np.nan)
    mean_s20 = np.full_like(dsf, np.nan)
    n_used = np.full((B, Y, X), -32768, dtype=np.int16)
    n_total = np.full((B, Y, X), -32768, dtype=np.int16)

    for b in range(B):
        lower = DEFICIT_BIN_EDGES[b]
        upper = DEFICIT_BIN_EDGES[b + 1]
        if b == B - 1:
            mask = valid_q20 & (S20 >= lower) & (S20 <= upper)
        else:
            mask = valid_q20 & (S20 >= lower) & (S20 < upper)

        dsf[b], n_used[b], n_total[b] = weighted_dsf(
            Qbm, Qm, mask, MIN_DEFICIT_BIN_MONTHS
        )
        bbi[b] = (dsf[b] - BFI).astype(np.float32)
        brr[b] = baseflow_retention_ratio(
            Qbm, expected_qb, mask, MIN_DEFICIT_BIN_MONTHS
        )

        valid_s = mask & np.isfinite(S20) & (Qm > EPS)
        count_s = np.sum(valid_s, axis=0)
        sum_s = np.sum(np.where(valid_s, S20, 0.0), axis=0, dtype=np.float64)
        mean = safe_ratio(sum_s, count_s.astype(np.float64))
        mean[count_s < MIN_DEFICIT_BIN_MONTHS] = np.nan
        mean_s20[b] = mean

    # Trend of volume-weighted BBI across populated S20 bins.
    valid_bin = np.isfinite(mean_s20) & np.isfinite(bbi)
    n_bins = np.sum(valid_bin, axis=0).astype(np.int8)
    x = np.where(valid_bin, mean_s20, 0.0).astype(np.float64)
    y = np.where(valid_bin, bbi, 0.0).astype(np.float64)
    n_float = n_bins.astype(np.float64)
    sum_x = np.sum(x, axis=0)
    sum_y = np.sum(y, axis=0)
    sum_xx = np.sum(x * x, axis=0)
    sum_yy = np.sum(y * y, axis=0)
    sum_xy = np.sum(x * y, axis=0)
    numerator = n_float * sum_xy - sum_x * sum_y
    denom_x = n_float * sum_xx - sum_x * sum_x
    denom_y = n_float * sum_yy - sum_y * sum_y

    slope = np.full(BFI.shape, np.nan, dtype=np.float32)
    corr = np.full(BFI.shape, np.nan, dtype=np.float32)
    good_slope = (
        (n_bins >= MIN_VALID_DEFICIT_BINS_FOR_TREND)
        & (denom_x > EPS)
        & np.isfinite(BFI)
    )
    slope[good_slope] = (numerator[good_slope] / denom_x[good_slope]).astype(
        np.float32
    )
    good_corr = good_slope & (denom_y > EPS)
    corr[good_corr] = (
        numerator[good_corr]
        / np.sqrt(denom_x[good_corr] * denom_y[good_corr])
    ).astype(np.float32)

    result = {
        "DSF_deficit_bin": dsf,
        "BBI_add_deficit_bin": bbi,
        "BRR_deficit_bin": brr,
        "mean_S20_deficit_bin": mean_s20,
        "n_months_used_deficit_bin": n_used,
        "n_months_total_deficit_bin": n_total,
        "BBI_deficit_bin_slope": slope,
        "BBI_deficit_bin_pearson_r": corr,
        "n_valid_deficit_bins": n_bins,
    }
    return result

# NETCDF OUTPUT

def _create_float(
    nc: Dataset,
    name: str,
    dims: tuple[str, ...],
    long_name: str,
    units: str = "1",
):
    var = nc.createVariable(
        name,
        "f4",
        dims,
        zlib=True,
        complevel=COMPLEVEL,
        fill_value=np.nan,
    )
    var.long_name = long_name
    var.units = units
    return var


def _create_int16(
    nc: Dataset,
    name: str,
    dims: tuple[str, ...],
    long_name: str,
):
    var = nc.createVariable(
        name,
        "i2",
        dims,
        zlib=True,
        complevel=COMPLEVEL,
        fill_value=-32768,
    )
    var.long_name = long_name
    return var


def create_output_netcdf(lat: np.ndarray, lon: np.ndarray) -> None:
    if OUT_NC.exists():
        OUT_NC.unlink()

    with Dataset(OUT_NC, "w", format="NETCDF4") as nc:
        nc.createDimension("lat", lat.size)
        nc.createDimension("lon", lon.size)
        nc.createDimension("cal_month", 12)
        nc.createDimension("threshold", len(THRESHOLD_PCTS))
        nc.createDimension("severity_class", len(SEVERITY_CLASS_NAMES))
        nc.createDimension("deficit_bin", len(DEFICIT_BIN_LABELS))

        vlat = nc.createVariable("lat", "f8", ("lat",))
        vlon = nc.createVariable("lon", "f8", ("lon",))
        vlat[:] = lat
        vlon[:] = lon


        vcal = nc.createVariable("cal_month", "i1", ("cal_month",))
        vcal[:] = np.arange(1, 13, dtype=np.int8)

        vthr = nc.createVariable("threshold_pct", "i2", ("threshold",))
        vthr[:] = THRESHOLD_PCTS

        vsev = nc.createVariable("severity_class", str, ("severity_class",))
        vsev[:] = np.asarray(SEVERITY_CLASS_NAMES, dtype=object)

        vbin = nc.createVariable("deficit_bin_label", str, ("deficit_bin",))
        vbin[:] = np.asarray(DEFICIT_BIN_LABELS, dtype=object)
        vlower = nc.createVariable("deficit_bin_lower", "f4", ("deficit_bin",))
        vupper = nc.createVariable("deficit_bin_upper", "f4", ("deficit_bin",))
        vlower[:] = DEFICIT_BIN_EDGES[:-1].astype(np.float32)
        vupper[:] = np.minimum(DEFICIT_BIN_EDGES[1:], 1.0).astype(np.float32)

        _create_float(
            nc,
            "Q_threshold",
            ("threshold", "cal_month", "lat", "lon"),
            "baseline calendar-month discharge percentile threshold",
            "input flow units aggregated per month",
        )
        _create_float(
            nc,
            "Qb_climatology_monthly",
            ("cal_month", "lat", "lon"),
            "baseline mean calendar-month baseflow from recommended-seven ensemble",
            "input flow units aggregated per month",
        )
        _create_float(
            nc,
            "BFI_clim_primary",
            ("lat", "lon"),
            "baseline BFI from recommended-seven ensemble median monthly baseflow",
        )
        _create_float(
            nc,
            "BFI_clim_legacy9",
            ("lat", "lon"),
            "baseline BFI from legacy-nine sensitivity ensemble",
        )
        _create_int16(
            nc,
            "n_baseline_months_BFI_primary",
            ("lat", "lon"),
            "number of valid baseline months used for primary BFI",
        )

        for method in PRIMARY_METHOD_NAMES:
            _create_float(
                nc,
                f"BFI_clim_{method}",
                ("lat", "lon"),
                f"baseline BFI from {method}",
            )

        threshold_dims = ("threshold", "lat", "lon")
        _create_float(nc, "DSF_threshold", threshold_dims, "volume-weighted DSF by threshold")
        _create_float(
            nc,
            "BBI_add_threshold",
            threshold_dims,
            "additive BBI by threshold: DSF minus BFI; primary metric",
        )
        _create_float(
            nc,
            "BBI_rel_threshold",
            threshold_dims,
            "relative partition shift by threshold: DSF divided by BFI minus one",
        )
        _create_float(nc, "BRR_threshold", threshold_dims, "absolute baseflow retention ratio by threshold")
        _create_int16(nc, "n_drought_months_used_threshold", threshold_dims, "positive-flow drought months used in DSF")
        _create_int16(nc, "n_drought_months_total_threshold", threshold_dims, "all drought months including zero-flow months")
        _create_float(nc, "drought_frequency_baseline", threshold_dims, "drought frequency in baseline period")
        _create_float(nc, "drought_frequency_analysis", threshold_dims, "drought frequency in analysis period")

        _create_float(nc, "DSF_threshold_legacy9", threshold_dims, "DSF by threshold using legacy-nine sensitivity ensemble")
        _create_float(nc, "BBI_add_threshold_legacy9", threshold_dims, "additive BBI by threshold using legacy-nine sensitivity ensemble")

        severity_dims = ("severity_class", "lat", "lon")
        _create_float(nc, "DSF_severity_class", severity_dims, "volume-weighted DSF for mutually exclusive severity classes")
        _create_float(nc, "BBI_add_severity_class", severity_dims, "additive BBI for mutually exclusive severity classes")
        _create_float(nc, "BRR_severity_class", severity_dims, "absolute baseflow retention ratio for severity classes")
        _create_int16(nc, "n_months_used_severity_class", severity_dims, "positive-flow months used in severity-class DSF")
        _create_int16(nc, "n_months_total_severity_class", severity_dims, "all months in each severity class")

        bin_dims = ("deficit_bin", "lat", "lon")
        _create_float(nc, "DSF_deficit_bin", bin_dims, "volume-weighted DSF within normalized-Q20 deficit bins")
        _create_float(nc, "BBI_add_deficit_bin", bin_dims, "additive BBI within normalized-Q20 deficit bins")
        _create_float(nc, "BRR_deficit_bin", bin_dims, "absolute baseflow retention ratio within normalized-Q20 deficit bins")
        _create_float(nc, "mean_S20_deficit_bin", bin_dims, "mean normalized Q20 deficit within each populated bin")
        _create_int16(nc, "n_months_used_deficit_bin", bin_dims, "positive-flow months used in deficit-bin DSF")
        _create_int16(nc, "n_months_total_deficit_bin", bin_dims, "all months in normalized-Q20 deficit bin")
        _create_float(nc, "BBI_deficit_bin_slope", ("lat", "lon"), "OLS slope of volume-weighted bin BBI against mean S20")
        _create_float(nc, "BBI_deficit_bin_pearson_r", ("lat", "lon"), "Pearson correlation of volume-weighted bin BBI with mean S20")
        _create_int16(nc, "n_valid_deficit_bins", ("lat", "lon"), "number of valid deficit bins used in trend")


        nc.title = "Drought-season BBI threshold and severity reanalysis"
        nc.source_file = str(IN_NC)
        nc.baseline_period = f"{BASELINE_START} to {BASELINE_END}"
        nc.analysis_period = f"{ANALYSIS_START} to {ANALYSIS_END}"
        nc.primary_BBI_definition = "BBI_add = DSF - BFI"
        nc.primary_DSF_definition = "DSF = sum(Qb) / sum(Q) over selected positive-flow drought months"
        nc.BRR_definition = "BRR = sum(actual Qb) / sum(calendar-month climatological Qb) for selected months"
        nc.primary_baseflow_ensemble = ", ".join(PRIMARY_METHOD_NAMES)
        nc.legacy_sensitivity_ensemble = ", ".join(LEGACY_METHOD_NAMES)
        nc.legacy_warning = "Legacy9-composition sensitivity includes two fixed-fraction members and duplicates the HYSEP sliding-interval series; all shared algorithms use the corrected implementations."
        nc.lyne_hollick_implementation = "Ladson-style endpoint reflection with forward-backward-forward filtering for the three-pass method"
        nc.lyne_hollick_reflection_days = int(LH_REFLECT_DAYS)
        nc.hysep_interval_days = int(HYSEP_N_DAYS)
        nc.hysep_interval_note = "A fixed 2N* interval of five days is used because catchment area is unavailable in the gridded input; change HYSEP_N_DAYS if a defensible spatially varying interval is available."
        nc.continuous_severity_definition = "S20 bins use volume-weighted DSF = sum(Qb)/sum(Q), not the mean of monthly Qb/Q fractions."
        nc.monthly_validity_rule = f"at least {MIN_DAILY_COVERAGE:.0%} finite daily values"
        nc.minimum_primary_methods = MIN_PRIMARY_METHODS
        nc.minimum_baseline_months_for_BFI = MIN_BASELINE_MONTHS_FOR_BFI
        nc.minimum_drought_months = MIN_DROUGHT_MONTHS
        nc.minimum_deficit_bin_months = MIN_DEFICIT_BIN_MONTHS
        nc.percentile_estimator = "NumPy linear percentile interpolation; drought condition uses Q < threshold"


def write_block(output: dict[str, np.ndarray], y0: int, y1: int, x0: int, x1: int) -> None:
    with Dataset(OUT_NC, "a") as nc:
        for name, values in output.items():
            var = nc.variables[name]
            if values.ndim == 4:
                var[:, :, y0:y1, x0:x1] = values
            elif values.ndim == 3:
                var[:, y0:y1, x0:x1] = values
            elif values.ndim == 2:
                var[y0:y1, x0:x1] = values
            else:
                raise ValueError(f"Unexpected rank for {name}: {values.ndim}")


# BLOCK CALCULATION

def process_block(
    Q_daily: np.ndarray,
    months_daily: pd.PeriodIndex,
    months_unique: pd.PeriodIndex,
    baseline_mask: np.ndarray,
    analysis_mask: np.ndarray,
) -> dict[str, np.ndarray]:
    Q_daily = Q_daily.astype(np.float32, copy=True)
    # Negative streamflow is nonphysical and must not be interpreted as drought.
    Q_daily[Q_daily < 0.0] = np.nan
    Qm = daily_to_monthly_sums(Q_daily, months_daily)

    monthly_methods: dict[str, np.ndarray] = {}
    method_bfi: dict[str, np.ndarray] = {}

    for method in PRIMARY_METHOD_NAMES:
        Qb_daily = compute_primary_method(Q_daily, method)
        Qbm = daily_to_monthly_sums(Qb_daily, months_daily)
        monthly_methods[method] = Qbm
        method_bfi[method], _ = calculate_bfi(Qbm, Qm, baseline_mask)
        del Qb_daily
        gc.collect()

    primary_stack = np.stack(
        [monthly_methods[m] for m in PRIMARY_METHOD_NAMES], axis=0
    )
    Qbm_primary, _ = ensemble_median(primary_stack, MIN_PRIMARY_METHODS)

    legacy_stack = np.stack(
        [
            monthly_methods["eckhardt"],
            monthly_methods["lyne_hollick_1pass"],
            monthly_methods["lyne_hollick_3pass"],
            monthly_methods["chapman"],
            0.5 * Qm,
            0.7 * Qm,
            monthly_methods["hysep_sliding_interval"],
            monthly_methods["hysep_fixed_interval"],
            monthly_methods["hysep_sliding_interval"],
        ],
        axis=0,
    ).astype(np.float32)
    Qbm_legacy, _ = ensemble_median(legacy_stack, MIN_LEGACY_METHODS)

    BFI_primary, n_bfi = calculate_bfi(Qbm_primary, Qm, baseline_mask)
    BFI_legacy, _ = calculate_bfi(Qbm_legacy, Qm, baseline_mask)

    thresholds = calendar_month_percentiles(Qm, months_unique, baseline_mask)
    threshold_map = {
        int(p): thresholds[k] for k, p in enumerate(THRESHOLD_PCTS)
    }

    qb_clim12 = calendar_month_mean(Qbm_primary, months_unique, baseline_mask)
    expected_qb = expand_calendar_month_field(qb_clim12, months_unique)

    n_thr = len(THRESHOLD_PCTS)
    _, Y, X = Qm.shape
    DSF = np.full((n_thr, Y, X), np.nan, dtype=np.float32)
    BBI = np.full_like(DSF, np.nan)
    BBI_rel = np.full_like(DSF, np.nan)
    BRR = np.full_like(DSF, np.nan)
    n_used = np.full((n_thr, Y, X), -32768, dtype=np.int16)
    n_total = np.full((n_thr, Y, X), -32768, dtype=np.int16)
    freq_base = np.full_like(DSF, np.nan)
    freq_analysis = np.full_like(DSF, np.nan)
    DSF_legacy = np.full_like(DSF, np.nan)
    BBI_legacy = np.full_like(DSF, np.nan)

    valid_primary = np.isfinite(Qm) & np.isfinite(Qbm_primary) & (Qm >= 0.0)
    valid_legacy = np.isfinite(Qm) & np.isfinite(Qbm_legacy) & (Qm >= 0.0)

    for k, pct in enumerate(THRESHOLD_PCTS):
        expanded = expand_calendar_month_field(thresholds[k], months_unique)
        threshold_valid = np.isfinite(expanded)
        drought_condition_primary = (
            valid_primary & threshold_valid & (Qm < expanded)
        )
        drought_condition_legacy = (
            valid_legacy & threshold_valid & (Qm < expanded)
        )
        mask_primary = analysis_mask[:, None, None] & drought_condition_primary
        mask_legacy = analysis_mask[:, None, None] & drought_condition_legacy
        mask_baseline = baseline_mask[:, None, None] & drought_condition_primary

        DSF[k], n_used[k], n_total[k] = weighted_dsf(
            Qbm_primary, Qm, mask_primary, MIN_DROUGHT_MONTHS
        )
        BBI[k] = (DSF[k] - BFI_primary).astype(np.float32)
        BBI_rel[k] = (safe_ratio(DSF[k], BFI_primary) - 1.0).astype(np.float32)
        BRR[k] = baseflow_retention_ratio(
            Qbm_primary, expected_qb, mask_primary, MIN_DROUGHT_MONTHS
        )

        eligible_baseline = (
            baseline_mask[:, None, None] & valid_primary & threshold_valid
        )
        eligible_analysis = (
            analysis_mask[:, None, None] & valid_primary & threshold_valid
        )
        freq_base[k] = drought_frequency(mask_baseline, eligible_baseline)
        freq_analysis[k] = drought_frequency(mask_primary, eligible_analysis)

        DSF_legacy[k], _, _ = weighted_dsf(
            Qbm_legacy, Qm, mask_legacy, MIN_DROUGHT_MONTHS
        )
        BBI_legacy[k] = (DSF_legacy[k] - BFI_legacy).astype(np.float32)

    severity_masks = make_severity_masks(
        Qm, months_unique, threshold_map, analysis_mask
    )
    S = len(SEVERITY_CLASS_NAMES)
    DSF_sev = np.full((S, Y, X), np.nan, dtype=np.float32)
    BBI_sev = np.full_like(DSF_sev, np.nan)
    BRR_sev = np.full_like(DSF_sev, np.nan)
    n_sev_used = np.full((S, Y, X), -32768, dtype=np.int16)
    n_sev_total = np.full((S, Y, X), -32768, dtype=np.int16)

    for s, name in enumerate(SEVERITY_CLASS_NAMES):
        DSF_sev[s], n_sev_used[s], n_sev_total[s] = weighted_dsf(
            Qbm_primary, Qm, severity_masks[name], MIN_DROUGHT_MONTHS
        )
        BBI_sev[s] = (DSF_sev[s] - BFI_primary).astype(np.float32)
        BRR_sev[s] = baseflow_retention_ratio(
            Qbm_primary, expected_qb, severity_masks[name], MIN_DROUGHT_MONTHS
        )

    deficit = deficit_bin_metrics(
        Qm,
        Qbm_primary,
        BFI_primary,
        expected_qb,
        months_unique,
        threshold_map[REFERENCE_THRESHOLD],
        analysis_mask,
    )

    output: dict[str, np.ndarray] = {
        "Q_threshold": thresholds,
        "Qb_climatology_monthly": qb_clim12,
        "BFI_clim_primary": BFI_primary,
        "BFI_clim_legacy9": BFI_legacy,
        "n_baseline_months_BFI_primary": n_bfi,
        "DSF_threshold": DSF,
        "BBI_add_threshold": BBI,
        "BBI_rel_threshold": BBI_rel,
        "BRR_threshold": BRR,
        "n_drought_months_used_threshold": n_used,
        "n_drought_months_total_threshold": n_total,
        "drought_frequency_baseline": freq_base,
        "drought_frequency_analysis": freq_analysis,
        "DSF_threshold_legacy9": DSF_legacy,
        "BBI_add_threshold_legacy9": BBI_legacy,
        "DSF_severity_class": DSF_sev,
        "BBI_add_severity_class": BBI_sev,
        "BRR_severity_class": BRR_sev,
        "n_months_used_severity_class": n_sev_used,
        "n_months_total_severity_class": n_sev_total,
    }
    output.update(deficit)
    for method, values in method_bfi.items():
        output[f"BFI_clim_{method}"] = values


    return output

# MAIN

def main() -> None:
    ensure_output_directory()

    ds = xr.open_dataset(IN_NC, decode_times=False)
    if FLOW_VAR not in ds:
        raise KeyError(f"{FLOW_VAR!r} not found in {IN_NC}")
    for name in (TIME_NAME, LAT_NAME, LON_NAME):
        if name not in ds:
            raise KeyError(f"Coordinate {name!r} not found in {IN_NC}")

    flow = ds[FLOW_VAR].transpose(TIME_NAME, LAT_NAME, LON_NAME)
    time = decode_time(ds)
    validate_time(time)

    months_daily = pd.PeriodIndex(time, freq="M")
    months_unique = months_daily.unique()
    baseline_mask = period_mask(months_unique, BASELINE_START, BASELINE_END)
    analysis_mask = period_mask(months_unique, ANALYSIS_START, ANALYSIS_END)

    if np.sum(baseline_mask) < MIN_BASELINE_MONTHS_FOR_BFI:
        raise ValueError("Baseline period is shorter than the minimum required for BFI.")
    if not np.any(analysis_mask):
        raise ValueError("Analysis period does not overlap the input data.")

    lat = np.asarray(ds[LAT_NAME].values)
    lon = np.asarray(ds[LON_NAME].values)
    create_output_netcdf(lat, lon)

    for y0 in range(0, lat.size, LAT_BLOCK):
        y1 = min(lat.size, y0 + LAT_BLOCK)
        for x0 in range(0, lon.size, LON_BLOCK):
            x1 = min(lon.size, x0 + LON_BLOCK)
            print(f"Processing lat[{y0}:{y1}] lon[{x0}:{x1}]")

            Q = flow.isel(
                {LAT_NAME: slice(y0, y1), LON_NAME: slice(x0, x1)}
            ).values.astype(np.float32)

            if not np.isfinite(Q).any():
                continue

            output = process_block(
                Q,
                months_daily,
                months_unique,
                baseline_mask,
                analysis_mask,
            )
            write_block(output, y0, y1, x0, x1)
            del Q, output
            gc.collect()

    ds.close()
    print(f"Completed. Output written to: {OUT_NC}")


if __name__ == "__main__":
    main()
